# Fix Cleaned Datasets (Phase 9-10 Repair, Production Grade)

This notebook performs a complete rebuild + cleaning + validation pipeline:
- `twcs_cleaned_fixed.csv`
- `questions_for_ml_fixed.csv`
- `labeled_questions_fixed.csv`

It includes:
- schema normalization and type coercion
- duplicate handling (ID-level + exact-row checks)
- missing-value handling by column type
- parent/child relationship consistency fixes
- robust text cleaning for RAG + ML
- strict validation gate before overwrite
- skewness diagnostics and optional transforms for Phase 11

Then optionally overwrites production files **only if all checks pass**.

> Important: CSV text may contain embedded newlines, so file line-count methods are unreliable for row counts.

In [1]:
# ============================================================
# 0) Setup
# ============================================================
from pathlib import Path
import re
import shutil
import numpy as np
import pandas as pd
from textblob import TextBlob

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

RAW_PATH = Path("../data/raw/twcs.csv")
ENH_PATH = Path("../data/processed/twcs_enhanced.csv")

CLEAN_DIR = Path("../data/cleaned")
LAB_DIR = Path("../data/labeled")
LOG_DIR = Path("../logs")
for d in (CLEAN_DIR, LAB_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

TWCS_FIXED = CLEAN_DIR / "twcs_cleaned_fixed.csv"
QML_FIXED = CLEAN_DIR / "questions_for_ml_fixed.csv"
LAB_FIXED = LAB_DIR / "labeled_questions_fixed.csv"
PH11_FEATURE_PROFILE = LOG_DIR / "phase11_feature_profile_from_fixed.csv"

for p in [RAW_PATH, ENH_PATH]:
    assert p.exists(), f"Missing required input file: {p.resolve()}"

print("✅ Setup complete")

✅ Setup complete


In [2]:
# ============================================================
# 1) Cleaning + labeling functions
# ============================================================
MENTION_RE = r"@[^\s@]+"
URL_PATTERNS = (
    r"(?i)https?://\S*",
    r"(?i)ftp://\S*",
    r"(?i)(?<![A-Za-z0-9_])www\.\S*",
    r"(?i)(?<![A-Za-z0-9_])t\.co/\S*",
    r"(?i)(?<![A-Za-z0-9_])t\.co\b",
)
EMOJI_RE = (
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+"
)

URGENCY_KEYWORDS = [
    "refund", "broken", "cancel", "down", "help",
    "urgent", "asap", "fix", "issue", "problem",
    "stuck", "error", "not working", "doesnt work",
    "wrong", "charged", "money", "scam", "fraud",
    "disappointed", "angry", "furious", "terrible",
    "worst", "useless", "hopeless",
]

def _strip_urls(series: pd.Series) -> pd.Series:
    s = series
    prev = None
    for _ in range(5):
        for pat in URL_PATTERNS:
            s = s.str.replace(pat, "", regex=True)
        if prev is not None and s.equals(prev):
            break
        prev = s.copy()
    return s

def clean_series_for_rag(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype("string")
    s = s.str.replace(MENTION_RE, "", regex=True)
    s = _strip_urls(s)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    # To satisfy strict downstream checks: no blanks.
    s = s.mask(s.str.len().fillna(0) == 0, "[EMPTY]")
    return s.astype(object)

def clean_series_for_ml(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype("string")
    s = s.str.replace(MENTION_RE, "", regex=True)
    s = _strip_urls(s)
    s = s.str.replace(EMOJI_RE, "", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.lower()
    s = s.mask(s.str.len().fillna(0) == 0, "[EMPTY]")
    return s.astype(object)

print("✅ Cleaning functions ready")

✅ Cleaning functions ready


In [3]:
# ============================================================
# 2) Load source data and verify enhanced base
# ============================================================
print("Loading enhanced dataset (full)...")
df_enh = pd.read_csv(ENH_PATH, low_memory=False)
df_enh.columns = [c.strip() for c in df_enh.columns]

print(f"Enhanced rows: {len(df_enh):,}")
print(f"Enhanced columns: {df_enh.shape[1]}")

expected_rows = 2_811_774
assert len(df_enh) == expected_rows, f"Enhanced row mismatch: {len(df_enh):,} != {expected_rows:,}"
assert "tweet_id" in df_enh.columns, "tweet_id missing"
assert df_enh["tweet_id"].is_unique, "tweet_id not unique in enhanced dataset"

# Basic type normalization for core columns
for col in ["tweet_id", "conversation_id", "position_in_conversation", "total_tweets_in_conversation"]:
    if col in df_enh.columns:
        df_enh[col] = pd.to_numeric(df_enh[col], errors="coerce")

for bcol in ["inbound", "is_question", "has_response"]:
    if bcol in df_enh.columns:
        df_enh[bcol] = df_enh[bcol].astype("boolean")

# Missing handling by type: conservative and explicit
# - text-like: keep source in text_original, use placeholders in derived columns later
# - numeric IDs/positions: do not synthesize unknown ids; keep as NaN if truly missing
# - boolean flags: fill missing with False for consistency where present
for bcol in ["inbound", "is_question", "has_response"]:
    if bcol in df_enh.columns:
        df_enh[bcol] = df_enh[bcol].fillna(False)

assert df_enh["conversation_id"].isna().sum() == 0, "conversation_id has nulls"

print("✅ Enhanced dataset base checks passed")

Loading enhanced dataset (full)...
Enhanced rows: 2,811,774
Enhanced columns: 14
✅ Enhanced dataset base checks passed


In [8]:
# ============================================================
# 3) Rebuild twcs_cleaned_fixed.csv (complete cleaning)
# ============================================================
df_twcs = df_enh.copy()

# Preserve source text before any transform
if "text" not in df_twcs.columns:
    raise ValueError("Enhanced dataset must contain 'text' column")

df_twcs["text_original"] = df_twcs["text"]
df_twcs["text_for_rag"] = clean_series_for_rag(df_twcs["text"])
df_twcs["text_for_ml"] = clean_series_for_ml(df_twcs["text"])

# ID-level deduplication (authoritative key)
before = len(df_twcs)
df_twcs = df_twcs.drop_duplicates(subset=["tweet_id"], keep="first").copy()
removed_id_dups = before - len(df_twcs)
print(f"ID dedup removed rows: {removed_id_dups:,}")

# Exact-row duplicate audit (report only; should be 0 after id dedup)
exact_dups = int(df_twcs.duplicated().sum())
print(f"Exact duplicate rows remaining: {exact_dups:,}")

# Parent reference handling: keep row count stable; null-out invalid refs
if "parent_tweet_id" in df_twcs.columns:
    ids = set(df_twcs["tweet_id"].astype(int).tolist())
    parent = pd.to_numeric(df_twcs["parent_tweet_id"], errors="coerce")
    valid_parent_mask = parent.isna() | parent.isin(ids)
    bad_parent = int((~valid_parent_mask).sum())
    if bad_parent > 0:
        df_twcs.loc[~valid_parent_mask, "parent_tweet_id"] = np.nan
    print(f"Invalid parent_tweet_id fixed (set NaN): {bad_parent:,}")

# response_tweet_ids hygiene (keep only ids that exist)
if "response_tweet_ids" in df_twcs.columns:
    ids = set(df_twcs["tweet_id"].astype(int).tolist())

    def sanitize_children(val):
        if pd.isna(val) or str(val).strip() == "":
            return ""
        toks = [t.strip() for t in str(val).split(",") if t.strip()]
        keep = []
        for t in toks:
            try:
                x = int(float(t))
                if x in ids:
                    keep.append(str(x))
            except Exception:
                continue
        return ",".join(keep)

    df_twcs["response_tweet_ids"] = df_twcs["response_tweet_ids"].map(sanitize_children)

    if "has_response" in df_twcs.columns:
        df_twcs["has_response"] = df_twcs["response_tweet_ids"].astype(str).str.strip().ne("")

# Keep row count invariant relative to enhanced expectation
assert len(df_twcs) == expected_rows, f"twcs_fixed row mismatch: {len(df_twcs):,}"
assert df_twcs["tweet_id"].is_unique, "tweet_id not unique after rebuild"

# Ensure strict no-blank/no-NA constraints on cleaned text columns
for col in ["text_for_rag", "text_for_ml"]:
    df_twcs[col] = df_twcs[col].fillna("[EMPTY]").astype(str)
    df_twcs[col] = df_twcs[col].mask(df_twcs[col].str.strip().eq(""), "[EMPTY]")

# Final defensive sweep (in case any transform reintroduced nulls)
df_twcs["text_for_ml"] = df_twcs["text_for_ml"].fillna("[EMPTY]")
df_twcs["text_for_rag"] = df_twcs["text_for_rag"].fillna("[EMPTY]")

df_twcs.to_csv(TWCS_FIXED, index=False)
print(f"✅ Saved {TWCS_FIXED.resolve()} ({len(df_twcs):,} rows)")

ID dedup removed rows: 0
Exact duplicate rows remaining: 0
Invalid parent_tweet_id fixed (set NaN): 24,945
✅ Saved C:\projects\decision-assistant\data\cleaned\twcs_cleaned_fixed.csv (2,811,774 rows)


In [9]:
# ============================================================
# 4) Rebuild questions_for_ml_fixed.csv
# ============================================================
qml = df_twcs.loc[df_twcs["is_question"] == True, [
    "tweet_id", "text_for_ml", "conversation_id", "author_id", "created_at"
]].copy()

qml = qml.rename(columns={
    "tweet_id": "question_id",
    "text_for_ml": "question_text_for_ml",
})

qml["question_text_for_ml"] = qml["question_text_for_ml"].fillna("[EMPTY]").astype(str)
qml["question_text_for_ml"] = qml["question_text_for_ml"].mask(
    qml["question_text_for_ml"].str.strip().eq(""), "[EMPTY]"
)
# Final defensive sweep for text NA/blanks
qml["question_text_for_ml"] = qml["question_text_for_ml"].fillna("[EMPTY]")
qml["question_text_for_ml"] = qml["question_text_for_ml"].mask(
    qml["question_text_for_ml"].astype(str).str.strip().eq(""), "[EMPTY]"
)

# Categorical/date-like stabilization (no row drops)
qml["author_id"] = qml["author_id"].fillna("unknown_author").astype(str)
qml["created_at"] = qml["created_at"].fillna("unknown_datetime").astype(str)

# Duplicate audits
dup_qid = int(qml["question_id"].duplicated().sum())
exact_dup_qml = int(qml.duplicated().sum())
text_dup_qml = int(qml["question_text_for_ml"].duplicated().sum())
print(f"Duplicate question_id rows: {dup_qid:,}")
print(f"Exact duplicate rows: {exact_dup_qml:,}")
print(f"Duplicate question_text_for_ml (semantic repeats): {text_dup_qml:,}")

expected_q = 1_537_843
print(f"questions_for_ml_fixed rows: {len(qml):,}")
assert len(qml) == expected_q, f"questions_for_ml row mismatch: {len(qml):,} != {expected_q:,}"
assert qml["question_id"].is_unique, "question_id not unique in questions_for_ml_fixed"
assert qml["question_text_for_ml"].isna().sum() == 0, "NA in question_text_for_ml"
assert qml["question_text_for_ml"].astype(str).str.strip().eq("").sum() == 0, "Blank string in question_text_for_ml"

qml.to_csv(QML_FIXED, index=False)
print(f"✅ Saved {QML_FIXED.resolve()} ({len(qml):,} rows)")

Duplicate question_id rows: 0
Exact duplicate rows: 0
Duplicate question_text_for_ml (semantic repeats): 87,471
questions_for_ml_fixed rows: 1,537,843
✅ Saved C:\projects\decision-assistant\data\cleaned\questions_for_ml_fixed.csv (1,537,843 rows)


In [10]:
# ============================================================
# 5) Rebuild labeled_questions_fixed.csv
# ============================================================
lab = qml.copy()
s = lab["question_text_for_ml"].fillna("[EMPTY]").astype(str)
is_empty = s.eq("[EMPTY]") | s.str.strip().eq("")

keyword_re = "|".join(re.escape(k) for k in URGENCY_KEYWORDS)
has_kw = s.str.contains(keyword_re, case=False, na=False, regex=True) & ~is_empty
has_excl = (s.str.count("!") >= 3) & ~is_empty

priority = pd.Series("normal", index=lab.index, dtype=object)
priority = priority.mask(has_kw, "urgent")
priority = priority.mask(has_excl & ~has_kw, "urgent")

# Sentiment only on remaining normal rows
need_sent = priority.eq("normal") & ~is_empty
idx = lab.index[need_sent]
CHUNK = 25_000
for j in range(0, len(idx), CHUNK):
    ci = idx[j:j+CHUNK]
    pol = s.loc[ci].map(lambda t: TextBlob(t).sentiment.polarity)
    priority.loc[pol.index[pol < -0.5]] = "urgent"

lab["priority"] = priority
assert lab["priority"].isna().sum() == 0, "Null priority found"
assert set(lab["priority"].unique()) <= {"urgent", "normal"}, "Invalid priority values"
assert len(lab) == expected_q, "labeled_questions row mismatch"

lab.to_csv(LAB_FIXED, index=False)
print(f"✅ Saved {LAB_FIXED.resolve()} ({len(lab):,} rows)")

✅ Saved C:\projects\decision-assistant\data\labeled\labeled_questions_fixed.csv (1,537,843 rows)


In [4]:
# ============================================================
# 6) Comprehensive validation
# ============================================================
# Self-contained imports/paths so this cell can run independently.
from pathlib import Path
import pandas as pd

CLEAN_DIR = Path("../data/cleaned")
LAB_DIR = Path("../data/labeled")
TWCS_FIXED = CLEAN_DIR / "twcs_cleaned_fixed.csv"
QML_FIXED = CLEAN_DIR / "questions_for_ml_fixed.csv"
LAB_FIXED = LAB_DIR / "labeled_questions_fixed.csv"

for p in [TWCS_FIXED, QML_FIXED, LAB_FIXED]:
    assert p.exists(), f"Missing fixed file: {p.resolve()}"

# Keep text tokens like 'NA' as literal strings (do not auto-convert to NaN)
twcs_fixed = pd.read_csv(TWCS_FIXED, low_memory=False, keep_default_na=False, na_filter=False)
qml_fixed = pd.read_csv(QML_FIXED, low_memory=False, keep_default_na=False, na_filter=False)
lab_fixed = pd.read_csv(LAB_FIXED, low_memory=False, keep_default_na=False, na_filter=False)

tweet_ids = set(twcs_fixed["tweet_id"].astype(int).tolist())
if "parent_tweet_id" in twcs_fixed.columns:
    p = pd.to_numeric(twcs_fixed["parent_tweet_id"], errors="coerce")
    bad_parent_refs = int((~(p.isna() | p.isin(tweet_ids))).sum())
else:
    bad_parent_refs = 0

url_probe = r"(?:https?://|t\.co/|\bwww\.)"
mention_probe = r"(?:^|[\s\n])@\w"
emoji_probe = r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U000024C2-\U0001F251]"

checks = {
    "twcs_cleaned rows": len(twcs_fixed) == 2_811_774,
    "twcs_cleaned unique tweet_id": twcs_fixed["tweet_id"].is_unique,
    "twcs_cleaned no NA in text_for_ml": twcs_fixed["text_for_ml"].isna().sum() == 0,
    "twcs_cleaned no blank text_for_ml": (twcs_fixed["text_for_ml"].astype(str).str.strip() == "").sum() == 0,
    "twcs_cleaned no blank text_for_rag": (twcs_fixed["text_for_rag"].astype(str).str.strip() == "").sum() == 0,
    "twcs_cleaned no @mentions in text_for_ml": twcs_fixed["text_for_ml"].astype(str).str.contains(mention_probe, regex=True, na=False).sum() == 0,
    "twcs_cleaned no @mentions in text_for_rag": twcs_fixed["text_for_rag"].astype(str).str.contains(mention_probe, regex=True, na=False).sum() == 0,
    "twcs_cleaned no URLs in text_for_ml": twcs_fixed["text_for_ml"].astype(str).str.contains(url_probe, case=False, regex=True, na=False).sum() == 0,
    "twcs_cleaned no URLs in text_for_rag": twcs_fixed["text_for_rag"].astype(str).str.contains(url_probe, case=False, regex=True, na=False).sum() == 0,
    "twcs_cleaned no emojis in text_for_ml": twcs_fixed["text_for_ml"].astype(str).str.contains(emoji_probe, regex=True, na=False).sum() == 0,
    "twcs_cleaned all parent IDs exist": bad_parent_refs == 0,
    "questions_for_ml rows": len(qml_fixed) == 1_537_843,
    "questions_for_ml no NA": qml_fixed["question_text_for_ml"].isna().sum() == 0,
    "questions_for_ml no blank": (qml_fixed["question_text_for_ml"].astype(str).str.strip() == "").sum() == 0,
    "labeled_questions rows": len(lab_fixed) == 1_537_843,
    "labeled_questions no null priority": lab_fixed["priority"].isna().sum() == 0,
    "labeled_questions valid priority": set(lab_fixed["priority"].unique()) <= {"urgent", "normal"},
}

print("=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)
for k, v in checks.items():
    print(("✅" if v else "❌") + f" {k}")

all_ok = all(checks.values())
print("\nOverall:", "✅ PASS" if all_ok else "❌ FAIL")

VALIDATION RESULTS
✅ twcs_cleaned rows
✅ twcs_cleaned unique tweet_id
✅ twcs_cleaned no NA in text_for_ml
✅ twcs_cleaned no blank text_for_ml
✅ twcs_cleaned no blank text_for_rag
✅ twcs_cleaned no @mentions in text_for_ml
✅ twcs_cleaned no @mentions in text_for_rag
✅ twcs_cleaned no URLs in text_for_ml
✅ twcs_cleaned no URLs in text_for_rag
✅ twcs_cleaned no emojis in text_for_ml
✅ twcs_cleaned all parent IDs exist
✅ questions_for_ml rows
✅ questions_for_ml no NA
✅ questions_for_ml no blank
✅ labeled_questions rows
✅ labeled_questions no null priority
✅ labeled_questions valid priority

Overall: ✅ PASS


In [6]:
# ============================================================
# 7) Skewness diagnostics + Phase 11 transform hints
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

# Ensure this cell can run independently
if "qml_fixed" not in globals():
    qml_fixed = pd.read_csv(Path("../data/cleaned/questions_for_ml_fixed.csv"), low_memory=False)
if "PH11_FEATURE_PROFILE" not in globals():
    PH11_FEATURE_PROFILE = Path("../logs/phase11_feature_profile_from_fixed.csv")

feat = pd.DataFrame({
    "char_count": qml_fixed["question_text_for_ml"].astype(str).str.len(),
    "word_count": qml_fixed["question_text_for_ml"].astype(str).str.split().str.len(),
    "exclamation_count": qml_fixed["question_text_for_ml"].astype(str).str.count("!"),
    "question_count": qml_fixed["question_text_for_ml"].astype(str).str.count(r"\?"),
})
letters = qml_fixed["question_text_for_ml"].astype(str).str.count(r"[a-zA-Z]")
caps = qml_fixed["question_text_for_ml"].astype(str).str.count(r"[A-Z]")
feat["caps_ratio"] = np.where(letters > 0, caps / letters, 0.0)

stats = feat.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T
stats["skew"] = feat.skew(numeric_only=True)
print("Feature distribution and skewness:")
display(stats[["mean", "50%", "90%", "95%", "99%", "max", "skew"]])

# Optional transformed features for linear models (not used to overwrite source text)
feat_tx = feat.copy()
for c in ["char_count", "word_count", "exclamation_count", "question_count"]:
    feat_tx[f"{c}_log1p"] = np.log1p(feat_tx[c])

profile = stats[["mean", "50%", "90%", "95%", "99%", "max", "skew"]].reset_index().rename(columns={"index": "feature"})
profile.to_csv(PH11_FEATURE_PROFILE, index=False)
print(f"✅ Saved feature profile: {PH11_FEATURE_PROFILE.resolve()}")
print("ℹ️ Keep raw counts for tree models; use log1p variants for linear/logistic models if skew is high.")

Feature distribution and skewness:


,mean,50%,90%,95%,99%,max,skew
char_count,92.227910,91.0,152.0,204.0,265.0,299.0,0.763331
word_count,17.253724,17.0,29.0,38.0,50.0,137.0,0.796667
exclamation_count,0.272931,0.0,1.0,2.0,4.0,238.0,34.381811
question_count,0.340604,0.0,1.0,2.0,3.0,280.0,57.752209
caps_ratio,0.013074,0.0,0.0,0.0,1.0,1.0,8.573223


✅ Saved feature profile: C:\projects\decision-assistant\logs\phase11_feature_profile_from_fixed.csv
ℹ️ Keep raw counts for tree models; use log1p variants for linear/logistic models if skew is high.


In [9]:
# ============================================================
# 8) Optional overwrite originals (ONLY after PASS)
# ============================================================
from pathlib import Path
import shutil
import pandas as pd

# Self-contained paths
CLEAN_DIR = Path("../data/cleaned")
LAB_DIR = Path("../data/labeled")
TWCS_FIXED = CLEAN_DIR / "twcs_cleaned_fixed.csv"
QML_FIXED = CLEAN_DIR / "questions_for_ml_fixed.csv"
LAB_FIXED = LAB_DIR / "labeled_questions_fixed.csv"

for p in [TWCS_FIXED, QML_FIXED, LAB_FIXED]:
    assert p.exists(), f"Missing fixed file: {p.resolve()}"

# If validation cell wasn't run in this kernel, derive a quick safety gate
if "all_ok" not in globals():
    tw = pd.read_csv(TWCS_FIXED, low_memory=False, keep_default_na=False, na_filter=False)
    qm = pd.read_csv(QML_FIXED, low_memory=False, keep_default_na=False, na_filter=False)
    lb = pd.read_csv(LAB_FIXED, low_memory=False, keep_default_na=False, na_filter=False)
    all_ok = (
        len(tw) == 2_811_774
        and tw["tweet_id"].is_unique
        and tw["text_for_ml"].isna().sum() == 0
        and (tw["text_for_ml"].astype(str).str.strip() == "").sum() == 0
        and len(qm) == 1_537_843
        and qm["question_text_for_ml"].isna().sum() == 0
        and (qm["question_text_for_ml"].astype(str).str.strip() == "").sum() == 0
        and len(lb) == 1_537_843
        and lb["priority"].isna().sum() == 0
        and set(lb["priority"].unique()) <= {"urgent", "normal"}
    )

OVERWRITE_ORIGINALS = True

if OVERWRITE_ORIGINALS:
    if not all_ok:
        raise RuntimeError("Validation failed. Will not overwrite originals.")

    shutil.copy2(TWCS_FIXED, CLEAN_DIR / "twcs_cleaned.csv")
    shutil.copy2(QML_FIXED, CLEAN_DIR / "questions_for_ml.csv")
    shutil.copy2(LAB_FIXED, LAB_DIR / "labeled_questions.csv")
    shutil.copy2(LAB_FIXED, CLEAN_DIR / "questions_labeled.csv")

    print("✅ Originals overwritten with fixed files.")
else:
    print("ℹ️ OVERWRITE_ORIGINALS is False (dry-run safe mode).")
    print("   If all checks passed, set OVERWRITE_ORIGINALS=True and re-run this cell.")

✅ Originals overwritten with fixed files.
